# Gemma-4-E4B Multimodal Reranker - Stage 2: +Audio (+Replay)\n\n3-stage QLoRA fine-tuning pipeline. Stage 2 trains on audio+text modalities while preserving stage 1 knowledge via Experience Replay (10% ratio).\n\n**Hardware**: Kaggle T4 (16GB VRAM)\n**Precision**: FP16 (T4 constraint)\n**Quantization**: QLoRA NF4 (bitsandbytes)\n

## 1. Setup & Dependencies

In [ ]:
!pip install -q \
    torch>=2.4 \
    transformers>=4.47 \
    peft>=0.13 \
    bitsandbytes>=0.44 \
    accelerate>=1.0 \
    safetensors>=0.4 \
    datasets>=3.0 \
    huggingface_hub[hf_xet] \
    mlflow>=2.18 \
    loguru>=0.7 \
    soundfile>=0.12 \
    numpy>=2.0 \
    av>=14.0 \
    pillow>=10.0

In [ ]:
import os
import json
import time
from pathlib import Path

import torch
import mlflow
from loguru import logger

# Kaggle Secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["MLFLOW_TRACKING_URI"] = secrets.get_secret("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = secrets.get_secret("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = secrets.get_secret("MLFLOW_TRACKING_PASSWORD")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

logger.info("CUDA available: {}", torch.cuda.is_available())
logger.info("GPU: {}", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
logger.info("VRAM: {:.1f} GB", torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0)

## 2. Configuration

In [ ]:
# --- Training Config ---
BASE_MODEL_ID = "google/gemma-4-E4B-it"
HUB_REPO_ID = "n24q02m/gemma4-e4b-reranker-v1"
STAGE = 2\n

## 3. Load Model (QLoRA NF4)

## 4. Prepare Data

Stage 1 uses:
- MS MARCO (text)
- NQ (text)
- FiQA (text)
- MMEB-train subset (text+image)
- VisualNews subset (text+image)

For initial testing, we use a small sample from MS MARCO.

In [ ]:
from datasets import load_dataset, Audio
import random

DATA_DIR.mkdir(parents=True, exist_ok=True)
train_path = DATA_DIR / "stage2_train.jsonl"
val_path = DATA_DIR / "stage2_val.jsonl"

logger.info("Preparing Stage 2 Data (Audio QA + 10% Stage 1 Replay)...")

# 1. Load Audio Data (Mock Spoken-SQuAD for pipeline)
ds_audio = load_dataset("facebook/multilingual_librispeech", "english", split="train[:500]")
samples = []
for row in ds_audio:
    query = "What is spoken in the audio?"
    # Using text transcription as document for mock mapping 
    doc_text = row["text"]
    samples.append({
        "query": query, "document": doc_text, "label": 1, 
        "modality": "audio", "audio_path": row["file"]
    })

# Add negative samples
for i in range(len(samples)):
    ns = samples[i].copy()
    ns["label"] = 0
    # Pick a random string for negative
    ns["document"] = samples[(i+1) % len(samples)]["document"]
    samples.append(ns)

# 2. Replay Buffer (10% old data) -> Assuming Stage 1 used text QA
replay_size = int(len(samples) * 0.10)
ds_text = load_dataset("microsoft/ms_marco", "v2.1", split=f"train[:{replay_size}]")
for row in ds_text:
    query = row["query"]
    passages = row.get("passages", {})
    if not passages or not passages.get("passage_text"): continue
    # just pick first passage as label=1
    samples.append({"query": query, "document": passages["passage_text"][0], "label": 1, "modality": "text"})

random.seed(42)
random.shuffle(samples)
val_size = max(1, int(len(samples) * 0.05))
train_samples, val_samples = samples[val_size:], samples[:val_size]

for p, data in [(train_path, train_samples), (val_path, val_samples)]:
    with open(p, "w") as f:
        for s in data:
            json.dump(s, f, ensure_ascii=False)
            f.write("\n

## 5. Dataset & DataLoader

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import soundfile as sf
import librosa

class RerankDataset(Dataset):
    def __init__(self, jsonl_path, processor, max_length=512, prefix=RERANKER_PREFIX):
        self.samples = [json.loads(l) for l in open(jsonl_path) if l.strip()]
        self.processor = processor
        self.max_length = max_length
        self.prefix = prefix
        self.sampling_rate = self.processor.feature_extractor.sampling_rate if hasattr(self.processor, "feature_extractor") else 16000

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        modality = s.get("modality", "text")
        
        content = []
        audio_array = None
        
        # Audio formatting
        if modality == "audio" and s.get("audio_path"):
            try:
                audio, sr = sf.read(s["audio_path"])
                # Resample if needed
                if sr != self.sampling_rate:
                    audio = librosa.resample(audio, orig_sr=sr, target_sr=self.sampling_rate)
                audio_array = audio
                content.append({"type": "audio", "audio": audio_array})
            except Exception as e:
                logger.warning(f"Failed to load audio {s[audio_path]}: {e}")

        # Text format (Document text)
        content.append({"type": "text", "text": f"<Query>\n{s[query]}\n</Query>\n<Document>\n{s[document]}\n

## 6. Training Helpers

In [ ]:
import bitsandbytes as bnb
from transformers import get_cosine_schedule_with_warmup


def find_lm_head(model):
    """Find lm_head in Gemma4 model (may be nested)."""
    base = model
    if hasattr(model, "base_model"):
        base = model.base_model
    if hasattr(base, "model"):
        base = base.model
    if hasattr(base, "lm_head") and isinstance(base.lm_head, torch.nn.Linear):
        return base.lm_head
    if hasattr(base, "language_model") and hasattr(base.language_model, "lm_head"):
        return base.language_model.lm_head
    raise AttributeError(f"Cannot find lm_head in {type(model).__name__}")


# Get token IDs
yes_id = processor.tokenizer.convert_tokens_to_ids("yes")
no_id = processor.tokenizer.convert_tokens_to_ids("no")
lm_head = find_lm_head(model)
logger.info("yes_id: {}, no_id: {}", yes_id, no_id)


def compute_loss(model, batch, yes_id, no_id, lm_head, kd_alpha=0.0, kd_temp=2.0):
    """Compute CE + optional KD loss."""
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        output_hidden_states=True,
    )
    hidden = outputs.hidden_states[-1]
    seq_lengths = batch["attention_mask"].sum(dim=-1) - 1
    last_h = hidden[torch.arange(hidden.size(0)), seq_lengths]
    logits = F.linear(last_h, lm_head.weight)
    logits_yn = torch.stack([logits[:, yes_id], logits[:, no_id]], dim=-1)

    loss_ce = F.cross_entropy(logits_yn, batch["labels"])
    metrics = {"loss_ce": loss_ce.item()}

    if kd_alpha > 0 and "teacher_scores" in batch:
        ts = batch["teacher_scores"]
        teacher_logits = torch.stack([ts, 1.0 - ts], dim=-1)
        teacher_probs = F.softmax(teacher_logits / kd_temp, dim=-1)
        student_lp = F.log_softmax(logits_yn / kd_temp, dim=-1)
        loss_kd = F.kl_div(student_lp, teacher_probs, reduction="batchmean")
        loss = (1 - kd_alpha) * loss_ce + kd_alpha * loss_kd * (kd_temp ** 2)
        metrics["loss_kd"] = loss_kd.item()
    else:
        loss = loss_ce
        metrics["loss_kd"] = 0.0

    metrics["loss_total"] = loss.item()
    return loss, metrics


# Optimizer
optimizer = bnb.optim.PagedAdamW8bit(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader) // GRAD_ACCUM * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = torch.amp.GradScaler("cuda")

logger.info("Total steps: {}, Warmup: {}", total_steps, warmup_steps)

## 7. MLflow Setup

In [ ]:
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", ""))
mlflow.set_experiment("gemma4-reranker")
run = mlflow.start_run(run_name=f"stage-{STAGE}")
mlflow.log_params({
    "stage": STAGE,
    "base_model": BASE_MODEL_ID,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
})
logger.info("MLflow run started: {}", run.info.run_id)

## 8. Training Loop

In [ ]:
global_step = 0
best_val_loss = float("inf")
patience = 0
PATIENCE_LIMIT = 3
LOG_EVERY = 10
EVAL_EVERY = 500  # steps

best_ckpt = OUTPUT_DIR / f"stage{STAGE}_best"

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_start = time.time()
    accum_loss = 0.0
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(train_loader):
        device = next(model.parameters()).device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.float16):
            loss, metrics = compute_loss(
                model, batch, yes_id, no_id, lm_head,
                kd_alpha=KD_ALPHA, kd_temp=KD_TEMPERATURE
            )
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        accum_loss += loss.item()

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            global_step += 1

            if global_step % LOG_EVERY == 0:
                lr = scheduler.get_last_lr()[0]
                vram = torch.cuda.memory_allocated() / 1e9
                logger.info(
                    "[E{} S{}] loss={:.4f} ce={:.4f} kd={:.4f} lr={:.2e} grad={:.2f} vram={:.1f}GB",
                    epoch, global_step, accum_loss * GRAD_ACCUM,
                    metrics["loss_ce"], metrics["loss_kd"],
                    lr, grad_norm.item() if isinstance(grad_norm, torch.Tensor) else grad_norm, vram
                )
                mlflow.log_metrics({
                    "train/loss": accum_loss * GRAD_ACCUM,
                    "train/loss_ce": metrics["loss_ce"],
                    "train/lr": lr,
                    "train/vram_gb": vram,
                }, step=global_step)

            accum_loss = 0.0

            # Eval
            if global_step % EVAL_EVERY == 0:
                model.eval()
                val_loss = 0.0
                val_count = 0
                with torch.no_grad():
                    for vb in val_loader:
                        vb = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in vb.items()}
                        with torch.amp.autocast("cuda", dtype=torch.float16):
                            vl, _ = compute_loss(model, vb, yes_id, no_id, lm_head)
                        val_loss += vl.item()
                        val_count += 1
                avg_val = val_loss / max(val_count, 1)
                logger.info("[Eval S{}] val_loss={:.4f}", global_step, avg_val)
                mlflow.log_metric("eval/val_loss", avg_val, step=global_step)

                if avg_val < best_val_loss:
                    best_val_loss = avg_val
                    patience = 0
                    best_ckpt.mkdir(parents=True, exist_ok=True)
                    model.save_pretrained(str(best_ckpt))
                    processor.save_pretrained(str(best_ckpt))
                    logger.info("Saved best checkpoint to {}", best_ckpt)
                else:
                    patience += 1

                model.train()

                if patience >= PATIENCE_LIMIT:
                    logger.info("Early stopping at step {}", global_step)
                    break

    epoch_time = time.time() - epoch_start
    logger.info("Epoch {} done in {:.0f}s", epoch, epoch_time)
    mlflow.log_metric("train/epoch_time_s", epoch_time, step=global_step)

    # Save epoch checkpoint
    epoch_ckpt = OUTPUT_DIR / f"stage{STAGE}_epoch{epoch}"
    epoch_ckpt.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(epoch_ckpt))

    if patience >= PATIENCE_LIMIT:
        break

mlflow.log_metric("eval/best_val_loss", best_val_loss, step=global_step)
logger.info("Training complete. Best val loss: {:.4f}", best_val_loss)

## 9. Save & Push

In [ ]:
# Save final adapter
final_adapter = OUTPUT_DIR / f"stage{STAGE}_final_adapter"
final_adapter.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_adapter))
processor.save_pretrained(str(final_adapter))
logger.info("Final adapter saved to {}", final_adapter)

# Push adapter to HF Hub (for Stage 2/3 to load)
model.push_to_hub(
    f"{HUB_REPO_ID}-stage{STAGE}-adapter",
    commit_message=f"feat: stage {STAGE} LoRA adapter",
    private=True,
)
processor.push_to_hub(f"{HUB_REPO_ID}-stage{STAGE}-adapter")
logger.info("Adapter pushed to HF Hub")

In [ ]:
mlflow.end_run()
logger.info("Stage {} complete!", STAGE)